# Feature Engineering & Dataset Preparation

This phase focuses on transforming raw logistics attributes into meaningful predictive features.

Key tasks include:

• Creating new derived variables  
• Transforming categorical variables using encoding  
• Selecting relevant features for modeling  
• Scaling numeric variables  
• Splitting data into training and testing sets

In [1]:
import pandas as pd
from pathlib import Path


from pathlib import Path

BASE = Path.cwd()

# Support running from the project root or notebooks/ folder
if not (BASE / "data").is_dir():
    if (BASE.parent / "data").is_dir():
        BASE = BASE.parent
    else:
        raise FileNotFoundError(
            "Project data folder not found. "
            "Run this notebook from the project root or notebooks folder."
        )

RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
DB = BASE / "data" / "logistics.db"

# Create only the directory needed for processed datasets
PROC.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE)
print("Raw data:", RAW)
print("Processed data:", PROC)


shipment = pd.read_csv(PROC / "shipment_cleaned.csv")
inventory = pd.read_csv(PROC / "inventory_cleaned.csv")
warehouse = pd.read_csv(PROC / "warehouse_cleaned.csv")

shipment.head()

Project root: d:\Athar\LogiTech_Logistics_Analysis
Raw data: d:\Athar\LogiTech_Logistics_Analysis\data\raw
Processed data: d:\Athar\LogiTech_Logistics_Analysis\data\processed


,Shipment_ID,Customer,Warehouse_ID,Destination_Region,Dispatch_Date,Delivery_Date,Delivery_Time_Days,Weight,Carrier_Name,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag
0,S008616,V55555555555555_8,PLANT03,PORT09,2023-01-01,2023-01-03,2,3.066737,V444_1,Delivered,AIR,1.5252,1,0
1,S007268,V555555555555555555_17,PLANT03,PORT09,2023-01-01,2023-01-03,2,4.680013,V444_1,Delivered,AIR,1.5252,1,0
2,S007608,V5555555_22,PLANT03,PORT09,2023-01-01,2023-01-03,2,5.145045,V444_1,Delivered,AIR,1.2020,1,0
3,S008205,V5555555_19,PLANT03,PORT09,2023-01-01,2023-01-03,2,5.939471,V444_1,Delivered,AIR,9.0884,1,0
4,S008717,V55555555555555_8,PLANT03,PORT09,2023-01-01,2023-01-03,2,19.270204,V444_1,Delivered,AIR,49.4272,1,0


## 2. Feature Creation

New features are created to capture operational relationships that may influence logistics performance.

In [2]:
shipment["Cost_Per_Weight"] = shipment["Shipment_Cost"] / shipment["Weight"]

shipment["Late_Delivery"] = (shipment["Delivery_Time_Days"] > 3).astype(int)

shipment[["Shipment_Cost","Weight","Cost_Per_Weight", "Late_Delivery"]].head()

,Shipment_Cost,Weight,Cost_Per_Weight,Late_Delivery
0,1.5252,3.066737,0.497336,0
1,1.5252,4.680013,0.325897,0
2,1.2020,5.145045,0.233623,0
3,9.0884,5.939471,1.530170,0
4,49.4272,19.270204,2.564955,0


## 3. Feature Transformation

Categorical variables are encoded into numeric form so they can be used in machine learning models.

In [3]:
shipment_encoded = pd.get_dummies(
    shipment,
    columns=["Destination_Region","Carrier_Name"],
    drop_first=True
)

shipment_encoded.head()

,Shipment_ID,Customer,Warehouse_ID,Dispatch_Date,Delivery_Date,Delivery_Time_Days,Weight,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag,Cost_Per_Weight,Late_Delivery,Carrier_Name_V444_1,Carrier_Name_V44_3
0,S008616,V55555555555555_8,PLANT03,2023-01-01,2023-01-03,2,3.066737,Delivered,AIR,1.5252,1,0,0.497336,0,True,False
1,S007268,V555555555555555555_17,PLANT03,2023-01-01,2023-01-03,2,4.680013,Delivered,AIR,1.5252,1,0,0.325897,0,True,False
2,S007608,V5555555_22,PLANT03,2023-01-01,2023-01-03,2,5.145045,Delivered,AIR,1.2020,1,0,0.233623,0,True,False
3,S008205,V5555555_19,PLANT03,2023-01-01,2023-01-03,2,5.939471,Delivered,AIR,9.0884,1,0,1.530170,0,True,False
4,S008717,V55555555555555_8,PLANT03,2023-01-01,2023-01-03,2,19.270204,Delivered,AIR,49.4272,1,0,2.564955,0,True,False


## 4. Feature Selection

Irrelevant or redundant variables are removed to simplify the dataset and improve model performance.

In [4]:
features = shipment_encoded.drop(
    columns=["Shipment_ID","Customer","Dispatch_Date","Delivery_Date"]
)

features.head()

,Warehouse_ID,Delivery_Time_Days,Weight,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag,Cost_Per_Weight,Late_Delivery,Carrier_Name_V444_1,Carrier_Name_V44_3
0,PLANT03,2,3.066737,Delivered,AIR,1.5252,1,0,0.497336,0,True,False
1,PLANT03,2,4.680013,Delivered,AIR,1.5252,1,0,0.325897,0,True,False
2,PLANT03,2,5.145045,Delivered,AIR,1.2020,1,0,0.233623,0,True,False
3,PLANT03,2,5.939471,Delivered,AIR,9.0884,1,0,1.530170,0,True,False
4,PLANT03,2,19.270204,Delivered,AIR,49.4272,1,0,2.564955,0,True,False


## 5. Feature Scaling

Numeric variables are standardized to ensure consistent scale across model inputs.

In [5]:
import numpy as np

from sklearn.preprocessing import StandardScaler

features.replace([np.inf, -np.inf], np.nan, inplace=True)
features.fillna(features.median(numeric_only=True), inplace=True)

print("Cleaned dataset shape:", features.shape)

scaler = StandardScaler()

numeric_cols = features.select_dtypes(include="number").columns
numeric_cols = numeric_cols.drop("Late_Delivery")

features[numeric_cols] = scaler.fit_transform(features[numeric_cols])
features.head()

Cleaned dataset shape: (9215, 12)


,Warehouse_ID,Delivery_Time_Days,Weight,Shipment_Status,Delivery_Mode,Shipment_Cost,Delivery_Accuracy_Flag,Damage_Flag,Cost_Per_Weight,Late_Delivery,Carrier_Name_V444_1,Carrier_Name_V44_3
0,PLANT03,-1.344872,-0.252458,Delivered,AIR,-0.033860,0.145873,-0.166304,-0.082586,0,True,False
1,PLANT03,-1.344872,-0.228222,Delivered,AIR,-0.033860,0.145873,-0.166304,-0.084655,0,True,False
2,PLANT03,-1.344872,-0.221236,Delivered,AIR,-0.034783,0.145873,-0.166304,-0.085768,0,True,False
3,PLANT03,-1.344872,-0.209301,Delivered,AIR,-0.012242,0.145873,-0.166304,-0.070124,0,True,False
4,PLANT03,-1.344872,-0.009036,Delivered,AIR,0.103057,0.145873,-0.166304,-0.057639,0,True,False


## 7. Saving Engineered Dataset

The processed dataset is saved for use in the modeling phase.

In [6]:
features.to_csv(PROC / "model_ready_dataset.csv", index=False)

print("Feature engineered dataset saved.")

Feature engineered dataset saved.


## Feature Engineering Summary

The dataset has been prepared for predictive modeling through the following steps:

• Derived operational features such as cost per weight and late delivery indicators.  
• Encoded categorical variables for machine learning compatibility.  
• Removed non-informative identifiers and date columns.  
• Standardized numerical variables for consistent scaling.  
• Split the dataset into training and testing subsets.

This refined dataset is now ready for machine learning modeling in Phase 8.